# AI Usage Disclosure

An AI assistant (Claude) was used as a **code-writing helper** — boilerplate, docstrings, and
debugging. The research design, data, evaluation set, and conclusions are the author's own. All
code was reviewed, executed, and is fully understood by the author. Full note: `AI_USAGE.md`.


# הנגשת זכויות חברתיות באמצעות NLP
## Hebrew Semantic Retrieval for Social-Rights Questions

**Track 3 — Data-Driven NLP** · HIT · Dr. Igor Kleiner  
**Author:** Yanik Gotie (318553526)

**Research question:** Does sentence-embedding *semantic* retrieval (AlephBERT) beat classic
*lexical* retrieval (TF-IDF / BM25) for Hebrew social-rights questions — and does a **hybrid**
of the two beat both? The core difficulty is the gap between citizens' everyday phrasing
("מה מגיע לי אחרי לידה?") and the formal language of official sources.

This notebook runs the whole pipeline: **scrape → corpus → retrievers → evaluation → error analysis → figures**.
It reuses the modules in `src/` so the notebook and the CLI (`python src/run_all.py`) stay in sync.


## 0 · Setup

On **Google Colab**, run the cell below. It installs dependencies and makes the project's
`src/` package importable. Two ways to get the code into Colab — pick one:

* **A. GitHub:** set `REPO_URL` to your repo and run the clone cell.
* **B. Google Drive:** upload the project folder to Drive, mount it, and set `PROJECT_DIR`.

Running **locally** instead? Just set `PROJECT_DIR` to the repo root and skip the install/clone.


In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

# ---- Option A: clone from GitHub -------------------------------------------
REPO_URL = 'https://github.com/YanikGotie/hebrew-rights-retrieval.git'
# If the repo is PRIVATE, create a GitHub token (Settings > Developer settings >
# Personal access tokens, 'repo' scope) and paste it below. Leave '' if the repo
# is PUBLIC (recommended for a course project -> no token needed).
GITHUB_TOKEN = ''

# ---- Option B: a folder already on disk / Google Drive --------------------
PROJECT_DIR = os.getcwd()   # default: assume we are already in the repo root

if IN_COLAB and REPO_URL:
    url = REPO_URL
    if GITHUB_TOKEN:
        url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    PROJECT_DIR = '/content/project'
    if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
        subprocess.run(['rm', '-rf', PROJECT_DIR])
        r = subprocess.run(['git', 'clone', '-q', url, PROJECT_DIR])
        if r.returncode != 0:
            raise RuntimeError(
                'git clone failed. If the repo is PRIVATE, set GITHUB_TOKEN above, '
                'or make it PUBLIC on GitHub: Settings > General > Danger Zone > '
                'Change visibility > Public. (Option B: upload the folder to Google '
                'Drive, mount it, and set PROJECT_DIR.)'
            )

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
sys.path.insert(0, PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)


In [ ]:
# Dependencies (Colab). python-bidi gives correct right-to-left Hebrew in figures.
if IN_COLAB:
    !pip install -q sentence-transformers rank-bm25 python-bidi >/dev/null
print('dependencies ready')


## 1 · Data collection & corpus

Sources (public, see `DATA.md` / `ETHICS.md`):

* **Kol-Zchut** (`כל-זכות`) — fetched via the official **MediaWiki API** (clean plain-text
  extracts, CC-BY-SA), across six life-domains: disability, unemployment, health, family,
  income-support, parenting.
* **Bituach Leumi** (`המוסד לביטוח לאומי`) — a small curated set of benefit pages scraped
  from the `#mainContent` region (robots-aware, rate-limited).

Pages are cleaned (niqqud stripped, boilerplate removed) and **chunked** into passages.
The cell below uses the cached raw pulls if present, otherwise it scrapes live.


In [ ]:
import build_corpus, config
import pandas as pd, collections

if not os.path.exists(config.CORPUS_PATH):
    build_corpus.build()          # scrape + chunk (uses data/raw cache when available)
corpus = build_corpus.load_corpus()

n_docs = len({r['doc_id'] for r in corpus})
by_dom = collections.Counter(r['domain'] for r in corpus)
by_src = collections.Counter(r['source'] for r in corpus)
print(f'{len(corpus)} chunks from {n_docs} documents')
print('by source :', dict(by_src))
print('by domain :', dict(by_dom))
pd.DataFrame(corpus[:3])[['doc_id','title','domain','text']]


## 2 · Evaluation set

A hand-authored set of **40 colloquial Hebrew questions**, each mapped to its ground-truth
document(s). Each query carries a `phrasing_gap` tag (how far the wording is from the
source's formal language) used later in the error analysis. A retrieval is a *hit* when a
returned chunk's parent `doc_id` is in the gold set.


In [ ]:
import evaluate
queries = evaluate.load_eval()
print(f'{len(queries)} queries')
pd.DataFrame(queries)[['id','domain','phrasing_gap','query','relevant_doc_ids']].head(8)


## 3 · Retrievers

Five retrievers over a shared interface (`src/retrievers.py`):

1. **TF-IDF** — sparse lexical baseline (Hebrew-aware tokens + stopwords, cosine).
2. **BM25** — Okapi BM25 lexical baseline (bonus comparison).
3. **AlephBERT (semantic)** — Hebrew sentence embeddings, cosine over dense vectors.
4. **Hybrid (RRF)** — Reciprocal Rank Fusion of TF-IDF + semantic.
5. **Hybrid (weighted)** — min-max score fusion of TF-IDF + semantic.

The semantic model `imvladikon/sentence-transformers-alephbert` is downloaded from
HuggingFace on first use and the passage embeddings are cached to `data/processed/`.


In [ ]:
import retrievers as R
retrievers = R.build_all(corpus, include_bm25=True)   # downloads AlephBERT on first run
print('retrievers:', list(retrievers))


## 4 · Qualitative demo

Ask a free-text question and compare what each model returns. This is where the
formal-vs-colloquial gap becomes visible: the lexical models latch onto shared words,
while the semantic model matches meaning.


In [ ]:
def show(query, k=3):
    print('Q:', query)
    for name in ['tfidf','semantic','hybrid_rrf']:
        hits = retrievers[name].search(query, k)
        print(f'\n  [{name}]')
        for h in hits:
            print(f"    {h['rank']}. {h['title']}  ({h['doc_id']}, score={h['score']:.3f})")
    print('-'*70)

show('פיטרו אותי מהעבודה, מה מגיע לי?')
show('הילד שלי מתעכב בהתפתחות, לאן פונים?')


## 5 · Quantitative evaluation

Metrics: **Recall@k** (k = 1, 3, 5, 10), **MRR**, **nDCG@10**, **MAP** — written to
`results/metrics.csv` and `results/per_query.csv`.


In [ ]:
metrics_df, per_query_df = evaluate.run(retrievers)
metrics_df


## 5b · Train / validation / test split

This is retrieval, so the split is **query-level**: the corpus is the index (fit without labels), and the 40 labeled queries are split (stratified by domain, seed 42) into **validation/dev (14)** and **held-out test (26)**. No hyper-parameters are tuned on the labeled queries, so the split is an unbiased robustness check.


In [ ]:
import splits, eval_splits
splits.make_split()
split_df = eval_splits.run()
split_df


## 6 · Visualizations

Saved to `results/figures/` and shown inline below.


In [ ]:
import visualize
emb = getattr(retrievers['semantic'], 'embeddings', None)
visualize.run(metrics_df, per_query_df, corpus, emb)

from IPython.display import Image, display
for fig in ['metrics_bar.png','recall_curve.png','winloss_heatmap.png','embedding_tsne.png']:
    p = os.path.join(config.FIGURES_DIR, fig)
    if os.path.exists(p): display(Image(p))


## 7 · Error analysis

≥10 worked examples grouped into *semantic-wins*, *lexical-wins*, and *both-fail*, each with
an automatic diagnosis. Written to `results/error_analysis.md`.


In [ ]:
import error_analysis
error_analysis.build(retrievers, corpus)
from IPython.display import Markdown, display
display(Markdown(open(config.ERROR_ANALYSIS_PATH, encoding='utf-8').read()))


## 8 · Conclusions

Read the numbers off the table in §5 and the figures in §6:

* Compare **semantic vs. lexical** — does AlephBERT win on the high-`phrasing_gap` queries?
* Does the **hybrid** recover the best of both (it usually wins on MRR/nDCG)?
* The §7 error analysis explains *why* each model fails where it does.

See `reports/report.md` for the full write-up, and `REFLECTION.md` for the personal
reflection, failure log and work log.
